# Stage 1 — MobileViT-XS setup and smoke test

This notebook implements Track A1-A2 from the project plan. It installs the minimum model dependency, selects Apple MPS/CUDA/CPU automatically, loads the ImageNet-pretrained MobileViT-XS checkpoint from the Hugging Face Hub through timm, uses the checkpoint's own preprocessing, checks a binary forward pass, and records intermediate feature shapes.

No crack dataset is used yet. Passing this notebook proves only that the model pipeline works; it is not an experimental result.

## 1. Install the minimum dependency

Run this once per environment. timm installs its required PyTorch and torchvision dependencies when they are absent.

In [4]:
%pip install -q timm

## 2. Imports, seed, and device

The same notebook works on the local M4 Mac, a CUDA machine, or CPU.

In [6]:
import json
from pathlib import Path

import timm
import torch
from PIL import Image
from timm.data import create_transform, resolve_model_data_config

SEED = 42
MODEL_NAME = "mobilevit_xs.cvnets_in1k"
OUTPUT_DIR = Path("outputs/mobilevit_xs")

torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print({
    "python_device": str(DEVICE),
    "torch": torch.__version__,
    "timm": timm.__version__,
})

{'python_device': 'cuda', 'torch': '2.11.0+cu128', 'timm': '1.0.28'}


## 3. Load MobileViT-XS for binary classification

The pretrained ImageNet classifier is replaced with one output logit. The rest of the checkpoint remains pretrained.

In [7]:
model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=1,
)
model = model.to(DEVICE).eval()

total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
classifier_parameters = sum(parameter.numel() for parameter in model.get_classifier().parameters())
backbone_parameters = total_parameters - classifier_parameters

print(model.__class__.__name__)
print(f"Binary model parameters: {total_parameters:,}")
print(f"Backbone parameters: {backbone_parameters:,}")
print(f"One-logit head parameters: {classifier_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")
print("Note: the published 2.3M count includes the original 1,000-class ImageNet head.")

assert 1_000_000 < total_parameters < 3_000_000, total_parameters
assert model.num_classes == 1
assert getattr(model.get_classifier(), "out_features", None) == 1

ByobNet
Binary model parameters: 1,933,233
Backbone parameters: 1,932,848
One-logit head parameters: 385
Trainable parameters: 1,933,233
Note: the published 2.3M count includes the original 1,000-class ImageNet head.


## 4. Resolve the checkpoint's preprocessing

Do not manually guess resize, crop, normalization, interpolation, or channel handling. timm supplies the configuration attached to the checkpoint.

In [8]:
data_config = resolve_model_data_config(model)
inference_transform = create_transform(**data_config, is_training=False)

print(json.dumps({
    key: list(value) if isinstance(value, tuple) else value
    for key, value in data_config.items()
}, indent=2))
print(inference_transform)

{
  "input_size": [
    3,
    256,
    256
  ],
  "interpolation": "bicubic",
  "mean": [
    0.0,
    0.0,
    0.0
  ],
  "std": [
    1.0,
    1.0,
    1.0
  ],
  "crop_pct": 0.9,
  "crop_mode": "center"
}
Compose(
    Resize(size=284, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(256, 256))
    MaybeToTensor()
    Normalize(mean=tensor([0., 0., 0.]), std=tensor([1., 1., 1.]))
)


## 5. Deterministic binary forward pass

Two synthetic PIL images are passed through the model-specific transform. This exercises the same input route that real crack images will use in Stage 2.

In [9]:
images = [
    Image.new("RGB", (320, 280), color=(96, 96, 96)),
    Image.new("RGB", (280, 320), color=(160, 160, 160)),
]
batch = torch.stack([inference_transform(image) for image in images]).to(DEVICE)

with torch.inference_mode():
    logits = model(batch)

print("Input shape:", tuple(batch.shape))
print("Output shape:", tuple(logits.shape))
print("Logits:", logits.detach().cpu().flatten().tolist())

assert tuple(batch.shape) == (2, 3, 256, 256), tuple(batch.shape)
assert tuple(logits.shape) == (2, 1), tuple(logits.shape)
assert torch.isfinite(logits).all().item()

Input shape: (2, 3, 256, 256)
Output shape: (2, 1)
Logits: [0.058120984584093094, 0.055147480219602585]


## 6. Record intermediate feature maps

These shapes are the first presentation artifact and show how spatial resolution decreases while channel depth grows through MobileViT.

In [10]:
feature_model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    features_only=True,
).to(DEVICE).eval()

with torch.inference_mode():
    feature_maps = feature_model(batch)

feature_shapes = [list(feature.shape) for feature in feature_maps]
for index, shape in enumerate(feature_shapes, start=1):
    print(f"Stage {index}: {shape}")

assert feature_shapes
assert all(shape[0] == 2 for shape in feature_shapes)

Stage 1: [2, 32, 128, 128]
Stage 2: [2, 48, 64, 64]
Stage 3: [2, 64, 32, 32]
Stage 4: [2, 80, 16, 16]
Stage 5: [2, 384, 8, 8]


## 7. Save the Stage 1 summary

The JSON output can be reused in the presentation and checked into the research artifacts after the notebook passes.

In [11]:
summary = {
    "stage": "MobileViT-XS smoke test",
    "model_name": MODEL_NAME,
    "seed": SEED,
    "device": str(DEVICE),
    "input_shape": list(batch.shape),
    "output_shape": list(logits.shape),
    "total_parameters": total_parameters,
    "backbone_parameters": backbone_parameters,
    "classifier_parameters": classifier_parameters,
    "trainable_parameters": trainable_parameters,
    "feature_shapes": feature_shapes,
    "data_config": {
        key: list(value) if isinstance(value, tuple) else value
        for key, value in data_config.items()
    },
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = OUTPUT_DIR / "stage_one_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

print(f"Saved {summary_path.resolve()}")

Saved /content/outputs/mobilevit_xs/stage_one_summary.json


## Stage 1 exit criterion

Stage 1 is complete when every assertion passes and outputs/mobilevit_xs/stage_one_summary.json exists. The next notebook stage is SDNET2018 download, audit, class counts, and manifest creation.